# MTTV-FLP — Chat avec Qwen2.5-7B-Instruct (GPU T4)

**Modèle** : [`Qwen/Qwen2.5-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)

**Exécution** :
1. `Exécution` → `Modifier le type d'exécution` → `T4 GPU`
2. `Exécution` → `Tout exécuter`

**Contraintes** :
- Le modèle charge ~14 Go en float16 sur le T4 (16 Go VRAM)
- `device_map="cuda:0"` garantit que TOUT le modèle est sur GPU
- Si le GPU n'est pas activé, le notebook plante avec un message clair

---


In [ ]:
"""
CELLULE 1 — Installation des dépendances
Exécutée en premier pour éviter les ModuleNotFoundError.
"""
import subprocess
import sys

print("=" * 60)
print("CELLULE 1/5 — Installation des dépendances")
print("=" * 60)

DEPS = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + DEPS)

# Vérification
import importlib.metadata
for dep in ["transformers", "accelerate", "bitsandbytes", "torch"]:
    try:
        v = importlib.metadata.version(dep)
        print(f"  [OK] {dep}=={v}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  [FAIL] {dep} NON TROUVÉ")

print("[OK] Dépendances installées avec succès")


In [ ]:
"""
CELLULE 2 — Vérification GPU obligatoire
Plante avec un message clair si CUDA n'est pas disponible.
"""
import torch

print("=" * 60)
print("CELLULE 2/5 — Vérification GPU")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "!" * 60 + "\n"
        "  ERREUR : CUDA n'est pas disponible.\n"
        "  Le modèle Qwen2.5-7B-Instruct nécessite un GPU.\n"
        "\n"
        "  Solution :\n"
        "  1. Menu → Exécution → Modifier le type d'exécution\n"
        "  2. Sélectionner T4 GPU\n"
        "  3. Exécuter à nouveau cette cellule\n"
        "\n"
        "  Vérification : Exécutez !nvidia-smi pour confirmer\n"
        "!" * 60
    )

# Infos GPU
gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
vram_used = torch.cuda.memory_allocated(0) / (1024**3)

print(f"  GPU détecté     : {gpu_name}")
print(f"  VRAM totale     : {vram_total:.1f} Go")
print(f"  VRAM utilisée   : {vram_used:.2f} Go")
print(f"  CUDA version    : {torch.version.cuda}")
print(f"  PyTorch version : {torch.__version__}")
print("[OK] GPU prêt — chargement du modèle possible")


In [ ]:
"""
CELLULE 3 — Chargement du modèle Qwen2.5-7B-Instruct sur GPU

Points critiques :
  - device_map="cuda:0"  → force TOUT le modèle sur GPU (≠ "auto")
  - torch_dtype=torch.float16 → ~14 Go, tient dans 16 Go du T4
  - low_cpu_mem_usage=True → évite de dupliquer en RAM avant GPU
  - trust_remote_code=True → nécessaire pour Qwen2.5
"""
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=" * 60)
print("CELLULE 3/5 — Chargement du modèle")
print("=" * 60)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# ─── Tokenizer ────────────────────────────────────────────────────────
print(f"[1/3] Chargement du tokenizer {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
# Certains tokenizers Qwen n'ont pas de pad_token ; on utilise eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"  Tokenizer chargé en {time.time() - t0:.1f}s")
print(f"  Vocabulaire : {len(tokenizer)} tokens")
print(f"  Pad token   : {tokenizer.pad_token}")
print(f"  EOS token   : {tokenizer.eos_token}")

# ─── Modèle ───────────────────────────────────────────────────────────
print(f"[2/3] Chargement du modèle {MODEL_NAME} en float16 sur cuda:0...")
print(f"  (Ce chargement prend ~2-3 minutes sur T4)")
t0 = time.time()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda:0",           # ← CRITIQUE : force GPU
    torch_dtype=torch.float16,       # ← 14 Go au lieu de 28 Go
    trust_remote_code=True,          # ← nécessaire pour Qwen
    low_cpu_mem_usage=True,          # ← évite duplication CPU
)

load_time = time.time() - t0
print(f"[3/3] Vérification du chargement")

# --- Vérification du device ------------------------------------------
print(f"  Modèle chargé en {load_time:.1f}s")
print(f"  model.device        : {model.device}")

# Vérifier que tous les paramètres sont bien sur CUDA
n_on_cpu = sum(1 for p in model.parameters() if p.device.type == "cpu")
n_on_cuda = sum(1 for p in model.parameters() if p.device.type == "cuda")
total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"  Paramètres totaux   : {total_params:.2f}B")
print(f"  Couches sur CPU     : {n_on_cpu}")
print(f"  Couches sur CUDA    : {n_on_cuda}")

if n_on_cpu > 0:
    print("  ⚠️  Attention : certaines couches sont restées sur CPU !")
else:
    print(f"  ✅ Modèle entièrement sur GPU : {model.device}")

# --- VRAM après chargement -------------------------------------------
vram_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"  VRAM utilisée après chargement : {vram_after:.2f} Go / 16 Go")

# --- nvidia-smi ------------------------------------------------------
import subprocess as sp
print("\n  ─── nvidia-smi (VRAM) ───")
result = sp.run(
    ["nvidia-smi", "--query-gpu=memory.total,memory.used,memory.free",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
print(f"  {result.stdout.strip()}")

print("[OK] Modèle prêt pour l'inférence")


In [ ]:
"""
CELLULE 4 — Fonction generate() + exemple
"""

print("=" * 60)
print("CELLULE 4/5 — Fonction d'inférence + exemple")
print("=" * 60)


def generate(prompt: str, max_new_tokens: int = 512,
             temperature: float = 0.7, do_sample: bool = True) -> str:
    """
    Génère une réponse avec Qwen2.5-7B-Instruct.

    Args:
        prompt: Message utilisateur
        max_new_tokens: Nombre max de tokens à générer
        temperature: Contrôle de la créativité (0.0 = déterministe)
        do_sample: Activer l'échantillonnage (False = greedy)

    Returns:
        Réponse générée (texte brut, sans les tokens spéciaux)
    """
    # Format chat Qwen2.5
    messages = [
        {"role": "user", "content": prompt},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            encoded,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Décoder uniquement la partie générée (après le prompt)
    generated = outputs[0][encoded.shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return response


# ─── Exemple ──────────────────────────────────────────────────────────
print("Exemple d'inférence...")
print()

exemple_prompt = "Explique la photosynthèse à un enfant de 10 ans, en 3 phrases."
print(f"Prompt : {exemple_prompt}")
print()

t0 = time.time()
reponse = generate(exemple_prompt, max_new_tokens=256, temperature=0.7)
elapsed = time.time() - t0

print(f"Réponse :")
print(reponse)
print()
print(f"(généré en {elapsed:.1f}s — {len(reponse.split())} mots)")
print("[OK] Fonction generate() prête")


In [ ]:
"""
CELLULE 5 — Boucle de chat interactive

Tapez 'quit', 'exit' ou 'stop' pour quitter.
Tapez 'clear' pour effacer l'écran.
"""

print("=" * 60)
print("CELLULE 5/5 — Chat interactif")
print("=" * 60)
print("  Tapez 'quit' / 'exit' / 'stop' pour quitter")
print("  Tapez 'clear' pour effacer l'écran")
print("=" * 60)
print()

history = []

while True:
    try:
        user_input = input("\n🧑 Vous > ")
    except EOFError:
        print("\nAu revoir !")
        break

    if user_input.lower() in ("quit", "exit", "stop"):
        print("\nAu revoir !")
        break

    if user_input.lower() == "clear":
        import os as _os
        _os.system("cls" if _os.name == "nt" else "clear")
        continue

    if not user_input.strip():
        continue

    # Génération
    print("\n🤖 Assistant > ", end="", flush=True)
    t0 = time.time()
    try:
        response = generate(user_input, max_new_tokens=512, temperature=0.7)
        elapsed = time.time() - t0
        print(response)
        print(f"\n   ✨ {elapsed:.1f}s — {len(response.split())} mots")
        history.append((user_input, response))
    except Exception as e:
        print(f"\n  ❌ Erreur : {e}")
